# 08 - Grounded Generation
## Shakespeare Evidence Desk

- **Goal:** turn the cited, budgeted **context** from stage 07 into an answer written *strictly* from that evidence — citing the numbered passages, separating fact from interpretation, and **abstaining** when the corpus cannot support an answer.
- **Experiment axis:** the grounding contract — closed-book (no context) vs. permissive grounding vs. strict grounding with forced citations and principled abstention (the retrieval method, reranker, and context builder are held fixed; only the *prompt and answer rules* change).
- **Loads from Step 07:** `.artifacts/context-handoff.json` (the selected context builder), replayed on top of the retrieval, reranking, index, and chunk artifacts to rebuild each question's cited context.
- **Exports:** `.artifacts/generation-handoff.json` — the grounded-generation decision and the first end-to-end baseline for stages 09 (query handling) and 10 (continuous evaluation).


## Why This Stage

*In general — what grounded generation is and why a Retrieval-Augmented Generation (RAG) system exists for it.* Every stage so far has prepared *evidence*: ingestion cleaned it, chunking cut it, embedding and indexing made it searchable, retrieval found it, reranking ordered it, and context building packed it into one cited, budgeted block. None of them wrote an answer. Grounded generation is the stage that finally does — and the entire point of a RAG system is that this answer is written from the retrieved context, not from the model's parametric memory.

The core decision this stage owns is *the grounding contract*: the exact rules we give the model about what it may use, how it must cite, and when it must refuse. Hand the model a question with no context and it answers from memory — fluent, confident, and unverifiable (the hallucination this whole pipeline exists to prevent). Hand it context but ask politely to "use it" and it still drifts, blends in outside knowledge, and answers questions the corpus cannot support. Make the contract strict — answer only from the numbered passages, cite each claim, and abstain when the evidence is absent — and the answer becomes traceable and honest. Because stricter prompts cost tokens and can over-refuse, the choice is *measured*: we keep the simplest contract that never hallucinates on out-of-corpus questions, cites every claim it makes, and still answers the questions the corpus can support.


## Conceptual Foundations

### Grounding: answering from evidence, not from memory

A large language model carries a vast **parametric memory** — everything it absorbed in training. Ungrounded, it answers from that memory, which is why it can state falsehoods with total confidence and cite nothing. **Grounding** flips the source of truth: the answer must come from the **context** we assembled in stage 07, and every claim must be traceable to a numbered passage in it. The model stops being the source of facts and becomes the thing that *reads and composes* them. For a corpus of ten specific Shakespeare plays, that distinction is the whole product — "the Ghost names poison" is useful only if it comes from the actual passage, not from a half-remembered plot.

### Faithfulness versus fluency

A **faithful** answer says only what the context supports; a **fluent** answer merely reads well. Models are overwhelmingly good at fluency, and that is exactly the trap: a smooth, authoritative paragraph can be entirely unsupported. This stage optimizes for faithfulness first and treats fluency as necessary but not sufficient. The measurable shadow of faithfulness here is whether the answer's claims overlap the provided context and whether its citations resolve — a proxy we lean on until stage 10 brings a full judge.

### Citations: making claims checkable

Context building (stage 07) already numbered each passage and attached a resolvable **citation** (work, act and scene, speakers, `chunk_id`). Generation's job is to *reference those numbers* — `[1]`, `[2]` — next to the claims they support, never to invent citation strings. A citation the reader can follow back to the source line is what turns "trust me" into "check for yourself," and it is what makes the fact-versus-interpretation split enforceable.

### Abstention: the courage to say "not in the corpus"

The hardest and most valuable behavior is **abstention** — refusing to answer when the context does not contain the evidence. A system that always produces an answer will confidently answer "What is the population of Verona?" or "What advice does Prospero give in The Winter's Tale?" — questions whose premises the corpus cannot support — and every such answer is a hallucination. Principled abstention trades a little coverage (it will sometimes refuse a question it *could* have answered) for a large gain in trust. We make abstention an explicit, detectable output and measure how well each prompt earns it.

### Fact versus interpretation

Literary questions invite interpretation, and interpretation is where ungrounded claims sneak in. The grounding contract asks the model to **separate the two**: quote or paraphrase what a passage literally says (fact, cited), then clearly mark any reading built on top of it (interpretation). This keeps the cited core verifiable even when the surrounding analysis is debatable.

### What grounded generation is *not* (clean stage boundaries)

- **Not retrieval (05), reranking (06), or context building (07):** it does not search, reorder, or assemble evidence; it consumes the one cited context handed to it.
- **Not query handling (09):** it answers the question and context as given; it does not rewrite the query, route it, or decide whether to retrieve again.
- **Not evaluation (10):** it produces answers and a baseline; the systematic, judge-driven scoring of those answers is stage 10's job.

### Common misconceptions and failure modes

- **"A good model doesn't need strict instructions."** Capable models hallucinate *more* persuasively, not less. Without an explicit contract they blend parametric memory into grounded answers invisibly.
- **"Providing context is enough."** Context is necessary, not sufficient — a permissive prompt still lets the model ignore it, answer from memory, and skip citations. The rules do the work.
- **"Abstention means the system is weak."** The opposite: a system that never abstains cannot be trusted on anything, because it answers unanswerable questions with equal confidence.
- **"Citations guarantee truth."** A citation only guarantees *traceability*. A model can cite a passage that does not actually support the claim; that is why stage 10 judges faithfulness, not just citation presence.


## 1. Load the Previous Selection

- **Purpose:** start from stage 07's exported decision, not hidden kernel state. Grounded generation only *writes from* the context the pipeline already produced, so retrieval, reranking, and the context builder are fixed inputs rebuilt here from their handoffs.
- **What the code does:** locates the repository root; reads `context-handoff.json` (the selected builder, token budget, dedup threshold, parent-context policy) and, through it, the `reranking-handoff.json`, `retrieval-handoff.json`, `index.json`, and `chunks.json` it depends on; reconstructs the stage 05-07 pipeline (Best Matching 25 (BM25), dense, and hybrid Reciprocal Rank Fusion (RRF) retrieval, the selected reranker, and the selected context builder with deduplication and parent expansion); loads **every** labeled dev question — both answerable and unanswerable — because abstention can only be measured when out-of-corpus questions are present; and builds the cited **context** for each question once, so every generation arm answers from the *same* evidence.
- **Fails loudly, early:** if the context handoff or any upstream artifact is missing, run notebooks 04-07 first. If the index was built on offline-fallback vectors (`embedding_backend != 'hosted'`), the notebook **stops here** — grounding on fake retrieval is meaningless.
- **Expected result:** the selected context builder and budget, the fixed upstream configuration, the Azure chat and embedding deployments, how many answerable and unanswerable questions were loaded, and the average context size the generator will read. Question embeddings and any replayed reranking are served from the stage 03-06 disk caches; context assembly itself makes no model call.


In [ ]:
import hashlib
import json
import math
import re
from collections import defaultdict
from pathlib import Path

import tiktoken
from dotenv import dotenv_values
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from openai import AzureOpenAI

# Work from the repository root so the notebook and the package read the same .env and artifacts.
ROOT = Path.cwd().resolve()
if not (ROOT / '.env.example').is_file() and (ROOT.parent / '.env.example').is_file():
    ROOT = ROOT.parent
if not (ROOT / '.env.example').is_file():
    raise RuntimeError('Use the repository as the notebook working directory.')

ARTIFACTS = ROOT / '.artifacts'
context_handoff_path = ARTIFACTS / 'context-handoff.json'
index_path = ARTIFACTS / 'index.json'
retrieval_handoff_path = ARTIFACTS / 'retrieval-handoff.json'
reranking_handoff_path = ARTIFACTS / 'reranking-handoff.json'
chunks_path = ARTIFACTS / 'chunks.json'
questions_path = ARTIFACTS / 'eval-questions.json'
for required in (context_handoff_path, index_path, retrieval_handoff_path, reranking_handoff_path):
    if not required.is_file():
        raise RuntimeError(f'Missing .artifacts/{required.name}. Run notebooks 04-07 first.')

context_handoff = json.loads(context_handoff_path.read_text(encoding='utf-8'))
index_payload = json.loads(index_path.read_text(encoding='utf-8'))
retrieval_handoff = json.loads(retrieval_handoff_path.read_text(encoding='utf-8'))
reranking_handoff = json.loads(reranking_handoff_path.read_text(encoding='utf-8'))
chunks_payload = json.loads(chunks_path.read_text(encoding='utf-8')) if chunks_path.is_file() else {'chunks': []}
evaluation_questions = json.loads(questions_path.read_text(encoding='utf-8')) if questions_path.is_file() else {'development': [], 'heldout': []}

index_type = index_payload['index_type']
metric = index_payload['metric']
stored_dims = index_payload['dims']
embedding_backend = index_payload.get('embedding_backend', 'unknown')
records = index_payload['records']

# Stop now if upstream vectors are offline-fallback: grounding on fake retrieval is meaningless.
if embedding_backend != 'hosted':
    raise RuntimeError(
        f"index.json was built on the '{embedding_backend}' backend, not hosted Azure. "
        "Re-run notebooks 03-07 against Azure before generating answers.")

# ---- fixed upstream decisions this stage consumes (never re-decided here) ----
SELECTED_METHOD = retrieval_handoff['selected_method']
RRF_K = retrieval_handoff.get('rrf_k') or 60
bm25_params = retrieval_handoff.get('bm25_params', {'k1': 1.5, 'b': 0.75})
SELECTED_RERANKER = reranking_handoff['selected_reranker']
RERANK_POOL = reranking_handoff.get('rerank_pool', 20)
TOP_K = reranking_handoff.get('top_k', 5)
LLM_SNIPPET_CHARS = reranking_handoff.get('llm_reranker', {}).get('snippet_chars', 320)

SELECTED_BUILDER = context_handoff['selected_builder']          # raw | dedup-budget | dedup-parent
CONTEXT_TOKEN_BUDGET = context_handoff['token_budget']
CONTEXT_POOL = context_handoff.get('candidate_pool', RERANK_POOL)
DEDUP_THRESHOLD = context_handoff.get('dedup', {}).get('threshold', 0.6)
PARENT_NEIGHBORS = context_handoff.get('parent_context', {}).get('neighbors_each_side', 1)
PARENT_MAX_TOKENS = context_handoff.get('parent_context', {}).get('max_tokens', 700)

vectors = [r['embedding'] for r in records]
N = len(vectors)
work_titles = {r['work_id']: r['work'] for r in records}
encoder = tiktoken.get_encoding('cl100k_base')   # the same tokenizer the chunker used in notebook 02


def count_tokens(text):
    return len(encoder.encode(text))


def dot(a, b):
    # Score between two unit vectors; equals cosine similarity. Higher = more relevant.
    return sum(x * y for x, y in zip(a, b))


def l2_normalize(vector):
    length = math.sqrt(sum(value * value for value in vector))
    return [value / length for value in vector] if length else vector


_WORD_PATTERN = re.compile(r"[a-z0-9']+")


def tokenize(text):
    return _WORD_PATTERN.findall(text.lower())


# ---- replay stage 04 dense search + stage 05 BM25/RRF so the selected method rebuilds each pool ----
def exact_search_rows(query_vector, candidate_rows=None):
    pool = range(N) if candidate_rows is None else candidate_rows
    return sorted(pool, key=lambda i: dot(query_vector, vectors[i]), reverse=True)


if index_type == 'ivf':
    ivf = index_payload['ivf']
    ivf_centroids = ivf['centroids']
    ivf_lists = {int(cluster): rows for cluster, rows in ivf['lists'].items()}
    nprobe = index_payload['params']['nprobe']

    def dense_search(query_vector, k, candidate_rows=None):
        nearest = sorted(range(len(ivf_centroids)),
                         key=lambda c: dot(query_vector, ivf_centroids[c]), reverse=True)[:nprobe]
        pool = [i for c in nearest for i in ivf_lists.get(c, ())]
        if candidate_rows is not None:
            allowed = set(candidate_rows)
            pool = [i for i in pool if i in allowed]
        return exact_search_rows(query_vector, pool)[:k]
else:
    def dense_search(query_vector, k, candidate_rows=None):
        return exact_search_rows(query_vector, candidate_rows)[:k]


class BM25:
    """The same transparent Okapi BM25 keyword index from notebook 05, reused to replay its pool."""

    def __init__(self, documents, term_saturation=1.5, length_strength=0.75):
        self.k1 = term_saturation
        self.b = length_strength
        self.tokenized_docs = [tokenize(text) for text in documents]
        self.doc_lengths = [len(tokens) for tokens in self.tokenized_docs]
        self.avg_len = (sum(self.doc_lengths) / len(self.tokenized_docs)) if self.tokenized_docs else 0.0
        self.term_counts_per_doc = []
        document_frequency = defaultdict(int)
        for tokens in self.tokenized_docs:
            term_counts = defaultdict(int)
            for term in tokens:
                term_counts[term] += 1
            self.term_counts_per_doc.append(term_counts)
            for term in term_counts:
                document_frequency[term] += 1
        total_documents = len(self.tokenized_docs)
        self.idf = {
            term: math.log(1 + (total_documents - doc_count + 0.5) / (doc_count + 0.5))
            for term, doc_count in document_frequency.items()
        }

    def search(self, query, top_k, candidate_rows=None):
        query_terms = [term for term in tokenize(query) if term in self.idf]
        candidate_indices = range(len(self.tokenized_docs)) if candidate_rows is None else candidate_rows
        scored_rows = []
        for row_index in candidate_indices:
            term_counts = self.term_counts_per_doc[row_index]
            doc_length = self.doc_lengths[row_index]
            score = 0.0
            for term in query_terms:
                term_count = term_counts.get(term, 0)
                if not term_count:
                    continue
                length_normalizer = self.k1 * (1 - self.b + self.b * doc_length / (self.avg_len or 1))
                score += self.idf[term] * (term_count * (self.k1 + 1)) / (term_count + length_normalizer)
            if score > 0:
                scored_rows.append((row_index, score))
        scored_rows.sort(key=lambda row_and_score: row_and_score[1], reverse=True)
        return [row_index for row_index, _ in scored_rows[:top_k]]


bm25 = BM25([record['text'] for record in records],
            term_saturation=bm25_params.get('k1', 1.5), length_strength=bm25_params.get('b', 0.75))


def reciprocal_rank_fusion(rankings, damping, top_k):
    fused_scores = defaultdict(float)
    for ranking in rankings:
        for rank, row_index in enumerate(ranking):
            fused_scores[row_index] += 1.0 / (damping + rank + 1)
    ranked_rows = sorted(fused_scores.items(), key=lambda row_and_score: row_and_score[1], reverse=True)
    return [row_index for row_index, _ in ranked_rows[:top_k]]


def retrieve_pool(query_text, query_vector, pool_size):
    """Replay the stage 05 selected method to produce the candidate pool reranking then ordered."""
    if SELECTED_METHOD == 'bm25':
        return bm25.search(query_text, pool_size)
    if SELECTED_METHOD == 'dense':
        return dense_search(query_vector, pool_size)
    dense_ranking = dense_search(query_vector, pool_size)
    keyword_ranking = bm25.search(query_text, pool_size)
    return reciprocal_rank_fusion([dense_ranking, keyword_ranking], RRF_K, pool_size)


# ---- parent/neighbor lookup for the selected context builder ----
chunks_by_id = {c['chunk_id']: c for c in chunks_payload.get('chunks', [])}


def chunk_index(chunk_id):
    # chunk_id is 'work_id:strategy:index'; the trailing integer is the chunk's position within its work.
    tail = chunk_id.rsplit(':', 1)[-1]
    return int(tail) if tail.isdigit() else None


siblings_by_work = defaultdict(dict)
for chunk_id, chunk in chunks_by_id.items():
    position = chunk_index(chunk_id)
    if position is not None:
        siblings_by_work[chunk.get('work_id')][position] = chunk_id

# ---- Azure clients (keyless, Entra identity); same on-disk caches as notebooks 03-07 ----
config = {key: value for key, value in dotenv_values(ROOT / '.env').items() if value}
token_provider = get_bearer_token_provider(DefaultAzureCredential(), 'https://cognitiveservices.azure.com/.default')
client = AzureOpenAI(
    azure_endpoint=config['AZURE_OPENAI_ENDPOINT'],
    api_version=config['AZURE_OPENAI_API_VERSION'],
    azure_ad_token_provider=token_provider,
)
embedding_deployment = config['AZURE_OPENAI_EMBEDDING_DEPLOYMENT']
chat_deployment = config['AZURE_OPENAI_CHAT_DEPLOYMENT']

EMBED_CACHE = ARTIFACTS / 'embedding-cache'
RERANK_CACHE = ARTIFACTS / 'rerank-cache'
EMBED_CACHE.mkdir(parents=True, exist_ok=True)
RERANK_CACHE.mkdir(parents=True, exist_ok=True)
embed_api_calls = 0


def embed_query(text):
    """Embed one query, reusing the 03-07 on-disk cache; call Azure only on a cache miss."""
    global embed_api_calls
    cache_file = EMBED_CACHE / (hashlib.sha256(f'{embedding_deployment}|{text}'.encode('utf-8')).hexdigest() + '.json')
    if cache_file.is_file():
        full_vector = json.loads(cache_file.read_text())
    else:
        full_vector = client.embeddings.create(model=embedding_deployment, input=[text]).data[0].embedding
        cache_file.write_text(json.dumps(full_vector))
        embed_api_calls += 1
    return l2_normalize(full_vector[:stored_dims])


RERANK_SYSTEM = (
    'You are a retrieval reranker for a Shakespeare evidence desk. '
    'Given a question and a numbered list of candidate passages, order the candidates '
    'from most to least relevant to answering the question. Judge only the text shown; '
    'do not use outside knowledge and do not invent passages. '
    'Return a JSON object {"ranking": [numbers]} listing every candidate number exactly once, best first.'
)


def apply_selected_reranker(query_text, query_vector, candidate_rows):
    """Reorder the pool exactly as stage 06 decided. For the LLM arm, replay its disk-cached ordering."""
    if SELECTED_RERANKER == 'none':
        return list(candidate_rows)
    if SELECTED_RERANKER == 'semantic':
        return sorted(candidate_rows, key=lambda row: dot(query_vector, vectors[row]), reverse=True)
    pool = list(candidate_rows)
    cache_key = hashlib.sha256(
        f"{chat_deployment}|{LLM_SNIPPET_CHARS}|{query_text}|{','.join(map(str, pool))}".encode('utf-8')).hexdigest()
    cache_file = RERANK_CACHE / (cache_key + '.json')
    if cache_file.is_file():
        order = json.loads(cache_file.read_text())
    else:
        numbered = '\n'.join(
            f"[{position}] ({records[row]['work']}) " + ' '.join(records[row]['text'].split())[:LLM_SNIPPET_CHARS]
            for position, row in enumerate(pool))
        response = client.chat.completions.create(
            model=chat_deployment,
            response_format={'type': 'json_object'},
            messages=[{'role': 'system', 'content': RERANK_SYSTEM},
                      {'role': 'user', 'content': f'Question: {query_text}\n\nCandidates:\n{numbered}'}],
        )
        parsed = json.loads(re.search(r'\{.*\}', response.choices[0].message.content or '{}', re.DOTALL).group(0))
        order = [int(n) for n in parsed.get('ranking', []) if isinstance(n, (int, float))]
        cache_file.write_text(json.dumps(order))
    seen, reranked = set(), []
    for position in order:
        if 0 <= position < len(pool) and position not in seen:
            seen.add(position)
            reranked.append(pool[position])
    reranked.extend(pool[position] for position in range(len(pool)) if position not in seen)
    return reranked


# ---- the stage 07 context builder toolkit (dedup, parent expansion, citation assembly) ----
def token_set(text):
    return set(tokenize(text))


def jaccard(a, b):
    union = len(a | b)
    return len(a & b) / union if union else 0.0


def format_scene(scenes):
    if not scenes:
        return 'Unknown scene'
    act, scene = scenes[0]
    return f'Act {act}, Scene {scene}'


def parent_text(row):
    """Expand one chunk to its same-scene neighbors, overlap lines de-duplicated and the result token-capped."""
    record = records[row]
    work_id = record['work_id']
    position = chunk_index(record['chunk_id'])
    if position is None or work_id not in siblings_by_work:
        return record['text']
    pieces = []
    for offset in range(-PARENT_NEIGHBORS, PARENT_NEIGHBORS + 1):
        sibling_id = siblings_by_work[work_id].get(position + offset)
        sibling = chunks_by_id.get(sibling_id) if sibling_id else None
        if sibling and any(scene in record['scenes'] for scene in sibling.get('scenes', [])):
            pieces.append((position + offset, sibling['text']))
    pieces.sort()
    seen_lines, merged_lines = set(), []
    for _, text in pieces:
        for line in text.split('\n'):
            stripped = line.strip()
            if stripped and stripped in seen_lines:   # drop the overlap lines the chunker repeated
                continue
            if stripped:
                seen_lines.add(stripped)
            merged_lines.append(line)
    merged = '\n'.join(merged_lines).strip() or record['text']
    token_ids = encoder.encode(merged)
    return encoder.decode(token_ids[:PARENT_MAX_TOKENS]) if len(token_ids) > PARENT_MAX_TOKENS else merged


def passage_block(position, record, text):
    header = f"[{position}] {record['work']} - {format_scene(record['scenes'])}"
    if record['speakers']:
        header += f" ({', '.join(record['speakers'][:4])})"
    return f"{header}\n{text.strip()}"


def assemble(rows, texts):
    """Number the passages, build the context string, and attach a resolvable citation to each."""
    blocks, citations = [], []
    for position, (row, text) in enumerate(zip(rows, texts), start=1):
        record = records[row]
        blocks.append(passage_block(position, record, text))
        citations.append({
            'marker': position,
            'chunk_id': record['chunk_id'],
            'work': record['work'],
            'work_id': record['work_id'],
            'scene': format_scene(record['scenes']),
            'speakers': record['speakers'],
        })
    return {'rows': rows, 'context': '\n\n'.join(blocks), 'citations': citations}


def build_raw(ordered_rows):
    chosen = list(ordered_rows[:TOP_K])
    return assemble(chosen, [records[row]['text'] for row in chosen])


def _pack(ordered_rows, expand):
    kept, kept_signatures, texts, used_tokens = [], [], [], 0
    for row in ordered_rows:
        signature = token_set(records[row]['text'])
        if any(jaccard(signature, previous) >= DEDUP_THRESHOLD for previous in kept_signatures):
            continue
        text = parent_text(row) if expand else records[row]['text']
        block_tokens = count_tokens(passage_block(len(kept) + 1, records[row], text)) + 2
        if used_tokens + block_tokens > CONTEXT_TOKEN_BUDGET:
            continue
        kept.append(row)
        kept_signatures.append(signature)
        texts.append(text)
        used_tokens += block_tokens
    return assemble(kept, texts)


BUILDERS = {
    'raw': build_raw,
    'dedup-budget': lambda rows: _pack(rows, expand=False),
    'dedup-parent': lambda rows: _pack(rows, expand=True),
}
build_context = BUILDERS[SELECTED_BUILDER]


def context_for(question_text):
    """Replay retrieval + selected reranker + selected builder to produce one question's cited context."""
    query_vector = embed_query(question_text)
    pool = retrieve_pool(question_text, query_vector, CONTEXT_POOL)
    ordered = apply_selected_reranker(question_text, query_vector, pool)
    return build_context(ordered)


# ---- load EVERY dev question (answerable + unanswerable) so abstention can be measured ----
dev_questions = list(evaluation_questions.get('development', []))
question_contexts = []
for question in dev_questions:
    built = context_for(question['question'])
    question_contexts.append({
        'id': question.get('id'),
        'question': question['question'],
        'answerable': bool(question.get('answerable')),
        'expected_works': question.get('expected_works', []),
        'context': built['context'],
        'citations': built['citations'],
    })

answerable_count = sum(1 for q in question_contexts if q['answerable'])
unanswerable_count = len(question_contexts) - answerable_count
avg_context_tokens = sum(count_tokens(q['context']) for q in question_contexts) / max(len(question_contexts), 1)

print(f'Context builder: {SELECTED_BUILDER} | budget={CONTEXT_TOKEN_BUDGET} tokens | pool={CONTEXT_POOL}.')
print(f'Fixed upstream: retrieval={SELECTED_METHOD}, reranker={SELECTED_RERANKER}, top-k={TOP_K}.')
print(f'Azure deployments: chat={chat_deployment}, embedding={embedding_deployment}.')
print(f'Loaded {len(question_contexts)} dev question(s): {answerable_count} answerable, {unanswerable_count} unanswerable.')
print(f'Average context handed to the generator: {avg_context_tokens:.0f} tokens ({embed_api_calls} live embedding call(s)).')


## 2. Explain the Alternatives

A generator is **good** when it answers answerable questions from the evidence, cites the passages it uses, **abstains** on questions the corpus cannot support, and never smuggles in parametric memory. We **fix everything upstream** (retrieval method, reranker, context builder, and therefore the exact context) and vary only the **grounding contract** — the system prompt and answer rules.

### What we actually compare vs. what we only explain

| Approach | Role here | Why |
|---|---|---|
| **Closed-book (no context)** | ✅ **compared** (control) | the model answers from parametric memory alone — the hallucination baseline grounding must beat; it cannot cite our passages and cannot know what is out of corpus |
| **Permissive grounding** | ✅ **compared** | context is provided with a soft "use these passages" instruction, but no forced citation and no abstention rule — shows that context without rules is not enough |
| **Strict grounding + citations + abstention** | ✅ **compared** | answer only from the numbered passages, cite each claim as [n], separate fact from interpretation, and output a refusal sentinel when the evidence is absent |
| **Chain-of-thought / self-critique** | ○ **explained, not run** | the model reasons or critiques its own draft before answering; can help faithfulness but adds latency and tokens, and the gains are better measured by the judge in stage 10 |
| **Generate-then-verify (two-pass)** | ○ **explained, not run** | a second model call checks each claim against the context; stronger faithfulness at double the cost — a stage 10/12 reliability lever, not the baseline |
| **Structured / schema-constrained output** | ○ **explained, not run** | force a JSON answer with explicit claim-citation pairs; useful for downstream tooling but orthogonal to the grounding contract we are choosing here |

So three contracts are compared head-to-head; the rest are explained so the choice makes sense.

### How each arm actually works (a line or two)

- **Closed-book** — send only the question, no context. The model answers from memory. It is the honest yardstick for how much grounding actually changes, and it makes the hallucination on out-of-corpus questions visible.
- **Permissive grounding** — prepend the cited context and ask the model to use it, but impose no citation format and no refusal rule. The model usually answers, often well, but it drifts into outside knowledge and rarely abstains.
- **Strict grounding** — the full contract: answer strictly from the numbered passages, attach `[n]` citations, keep fact separate from interpretation, and emit `INSUFFICIENT_CONTEXT` when the passages cannot support an answer. More prompt, more discipline, detectable abstention.

### Why this is the experiment we run

The stage exists to make answers *trustworthy*, and trust has three measurable failure modes: answering from memory (ungrounded), failing to cite (unverifiable), and answering the unanswerable (hallucination). Closed-book versus grounded isolates the first; permissive versus strict isolates the second and third. The completion budget, the refusal sentinel, and the citation syntax are bounded settings, not competing arms.


## 3. Run Controlled Experiments

- **Fixed upstream:** the context for every question was assembled once in Section 1 with the selected builder; all three contracts answer from the *same* context, so any difference is the prompt, not the evidence.
- **What the code does:** defines the three grounding system prompts (closed-book, permissive, strict), a single `generate` call cached to disk by (deployment, contract, question, context) so a kernel restart replays answers instead of re-billing, a refusal sentinel (`INSUFFICIENT_CONTEXT`) the strict contract emits when the evidence is absent, and runs every contract over every dev question.
- **LLM calls:** each answer is one chat completion from the gpt-5 reasoning model (which uses its default sampling); the on-disk cache makes reruns reproducible, so on a warm cache there are **0** live calls. Closed-book sends no context; the grounded contracts send the Section 1 context.
- **Bounded by design:** a fixed completion budget (reasoning plus answer) and a fixed sentinel keep outputs parseable and cost bounded. Nothing here fabricates evidence — the context is copied verbatim from stage 07.
- **Expected result:** the three registered contracts, the number of answers produced per contract, how many were live calls versus cache hits, and a confirmation that all contracts saw identical contexts.


In [ ]:
import time

# gpt-5 reasoning models only accept the default temperature; the budget covers reasoning tokens plus the answer.
MAX_COMPLETION_TOKENS = 2000
ABSTAIN_SENTINEL = 'INSUFFICIENT_CONTEXT'   # the exact refusal token the strict contract must emit

CLOSED_BOOK_SYSTEM = (
    'You are a Shakespeare scholar. Answer the question as accurately as you can. '
    'Be concise.'
)

PERMISSIVE_SYSTEM = (
    'You are a Shakespeare evidence desk. Use the provided passages to help answer the question. '
    'Be concise.'
)

STRICT_SYSTEM = (
    'You are a Shakespeare evidence desk that answers strictly from provided passages.\n'
    'Rules:\n'
    '1. Use ONLY the numbered passages in the context. Do not use any outside knowledge.\n'
    '2. Cite every claim with the bracketed passage number(s) it comes from, e.g. [1] or [2][3].\n'
    '3. State facts the passages support first; clearly mark any interpretation as such.\n'
    f'4. If the passages do not contain enough information to answer, reply with exactly {ABSTAIN_SENTINEL} and nothing else.\n'
    'Be concise and do not invent passages, citations, speakers, or scenes.'
)

GEN_CONTRACTS = {
    'closed-book': {'system': CLOSED_BOOK_SYSTEM, 'use_context': False},
    'permissive': {'system': PERMISSIVE_SYSTEM, 'use_context': True},
    'strict': {'system': STRICT_SYSTEM, 'use_context': True},
}

GEN_CACHE = ARTIFACTS / 'generation-cache'
GEN_CACHE.mkdir(parents=True, exist_ok=True)
gen_api_calls = 0


def generate(contract_name, question_text, context_text):
    """One grounded answer for a (contract, question) pair, cached to disk and replayed on a cache miss."""
    global gen_api_calls
    contract = GEN_CONTRACTS[contract_name]
    user_message = (f'Context:\n{context_text}\n\nQuestion: {question_text}'
                    if contract['use_context'] else f'Question: {question_text}')
    cache_key = hashlib.sha256(
        f"{chat_deployment}|{contract_name}|{MAX_COMPLETION_TOKENS}|{user_message}".encode('utf-8')).hexdigest()
    cache_file = GEN_CACHE / (cache_key + '.json')
    if cache_file.is_file():
        cached = json.loads(cache_file.read_text())
        return cached['answer'], cached.get('latency_ms', 0.0)
    started = time.perf_counter()
    response = client.chat.completions.create(
        model=chat_deployment,
        max_completion_tokens=MAX_COMPLETION_TOKENS,
        messages=[{'role': 'system', 'content': contract['system']},
                  {'role': 'user', 'content': user_message}],
    )
    latency_ms = (time.perf_counter() - started) * 1000
    answer = (response.choices[0].message.content or '').strip()
    cache_file.write_text(json.dumps({'answer': answer, 'latency_ms': latency_ms}))
    gen_api_calls += 1
    return answer, latency_ms


generation_runs = {name: [] for name in GEN_CONTRACTS}
for name in GEN_CONTRACTS:
    for item in question_contexts:
        answer, latency_ms = generate(name, item['question'], item['context'])
        generation_runs[name].append({
            'id': item['id'],
            'question': item['question'],
            'answerable': item['answerable'],
            'expected_works': item['expected_works'],
            'context': item['context'],
            'citations': item['citations'],
            'answer': answer,
            'latency_ms': latency_ms,
        })

print(f'Registered contracts: {", ".join(GEN_CONTRACTS)}.')
print(f'Produced {len(question_contexts)} answer(s) per contract ({len(GEN_CONTRACTS) * len(question_contexts)} total).')
print(f'Live chat calls this run: {gen_api_calls} (rest served from the generation cache).')

print('All contracts answered from the identical Section 1 contexts.')

## Metrics in This Stage — What Each Number Means

Section 4 runs the three contracts over the same contexts and grades each answer against the question labels. The selection rule keeps the simplest contract that never hallucinates on out-of-corpus questions, cites every answer it gives, and still answers the questions the corpus supports.

| Metric | What it measures | How it is computed | What it signifies |
|---|---|---|---|
| Abstention accuracy | whether the system refuses when the corpus cannot answer | share of unanswerable questions on which the answer is the refusal sentinel | Primary safety constraint — any non-refusal here is a hallucination; must be 1.0. |
| Answer rate | whether the system answers the questions it should | share of answerable questions the system did not refuse | Over-refusal guard — abstaining on answerable questions wastes real evidence; higher is better. |
| Citation validity | whether every answered claim is traceable | share of answered questions whose bracketed markers all resolve to a passage in that context, with at least one citation | Grounding constraint — an answer with no resolvable citation cannot be checked; must be 1.0. |
| Grounding (lexical) | how much of the answer is drawn from the context | mean fraction of an answer's content words that appear in its context word set | Faithfulness proxy — low overlap suggests parametric drift; higher is better, pending the stage 10 judge. |
| Work relevance | whether the answer names the expected work(s) | share of answered answerable questions whose answer mentions every expected work title | Correctness proxy at the corpus level; higher is better. |
| Average tokens | how long the answers run | mean tokenized length of the produced answers (tiktoken cl100k_base) | Cost and verbosity indicator; read with grounding, not alone. |
| p95 latency (ms) | the per-answer generation cost | 95th percentile of the per-question call time | Each arm makes one chat call; lower is better. |


## 4. Measure and Inspect

- **The lenses:** **abstention accuracy** and **answer rate** (does it refuse the unanswerable and answer the answerable), **citation validity** and **grounding** (is each answer traceable and drawn from the evidence), **work relevance** (does it name the right play), and **tokens / latency** (cost).
- **Abstention accuracy — the primary safety metric.** Over the unanswerable questions, the fraction on which the system emits the refusal sentinel. Every non-refusal here is a confident answer to a question the corpus cannot support — a hallucination — so this gates eligibility, not merely scores it.
- **Citation validity is a hard constraint.** For each answered question we parse the bracketed markers and require that every one resolves to a passage in that question's context and that at least one citation is present. An answer no reader can trace back is ungrounded regardless of how right it sounds.
- **Grounding and work relevance are quality signals.** Grounding is the share of an answer's content words that appear in its context — a cheap, deterministic faithfulness proxy (the real judge arrives in stage 10). Work relevance checks that answered questions name every expected play, a coarse correctness check from the labels.
- **A worked answer and a worked refusal:** we print the strict contract's answer on a cross-play comparison question (citations visible) and its response to an out-of-corpus question (the refusal), so the behavior is visible by eye.
- **Honest limit:** the labeled dev set is tiny and the grounding proxy is lexical, so these numbers pre-screen contracts rather than certify quality. The systematic judge is stage 10.


In [ ]:
_STOPWORDS = {
    'the', 'a', 'an', 'and', 'or', 'but', 'of', 'to', 'in', 'on', 'for', 'with', 'as', 'by', 'at',
    'is', 'are', 'was', 'were', 'be', 'been', 'being', 'it', 'its', 'this', 'that', 'these', 'those',
    'he', 'she', 'they', 'his', 'her', 'their', 'who', 'what', 'which', 'how', 'does', 'do', 'did',
    'from', 'not', 'no', 'has', 'have', 'had', 'about', 'into', 'than', 'then', 'there', 'here',
}
_CITATION_PATTERN = re.compile(r'\[(\d+)\]')


def is_abstention(answer):
    return ABSTAIN_SENTINEL in answer.upper()


def cited_markers(answer):
    return [int(marker) for marker in _CITATION_PATTERN.findall(answer)]


def citations_resolve(answer, citations):
    # Valid when the answer cites at least one passage and every cited number exists in this context.
    markers = cited_markers(answer)
    valid_markers = {citation['marker'] for citation in citations}
    return bool(markers) and all(marker in valid_markers for marker in markers)


def grounding_overlap(answer, context_text):
    # Lexical faithfulness proxy: share of the answer's content words that appear in the context.
    answer_terms = {term for term in tokenize(_CITATION_PATTERN.sub(' ', answer)) if term not in _STOPWORDS}
    if not answer_terms:
        return 0.0
    context_terms = set(tokenize(context_text))
    return len(answer_terms & context_terms) / len(answer_terms)


def names_expected_works(answer, expected_works):
    lowered = answer.lower()
    return all(str(work).lower() in lowered for work in expected_works) if expected_works else True


def evaluate_contract(runs):
    """Score one contract: abstention, answer rate, citation validity, grounding, relevance, and cost."""
    answerable = [r for r in runs if r['answerable']]
    unanswerable = [r for r in runs if not r['answerable']]
    abstained_unanswerable = sum(1 for r in unanswerable if is_abstention(r['answer']))
    answered = [r for r in answerable if not is_abstention(r['answer'])]
    citation_valid = sum(1 for r in answered if citations_resolve(r['answer'], r['citations']))
    grounding = [grounding_overlap(r['answer'], r['context']) for r in answered]
    relevance = sum(1 for r in answered if names_expected_works(r['answer'], r['expected_works']))
    latencies = sorted(r['latency_ms'] for r in runs)
    tokens = [count_tokens(r['answer']) for r in runs]
    p95 = latencies[min(len(latencies) - 1, int(0.95 * len(latencies)))] if latencies else 0.0
    return {
        'abstention_accuracy': abstained_unanswerable / max(len(unanswerable), 1),
        'answer_rate': len(answered) / max(len(answerable), 1),
        'citation_validity': (citation_valid / len(answered)) if answered else 0.0,
        'grounding': (sum(grounding) / len(grounding)) if grounding else 0.0,
        'work_relevance': (relevance / len(answered)) if answered else 0.0,
        'avg_tokens': sum(tokens) / max(len(tokens), 1),
        'p95_latency_ms': p95,
    }


generation_metrics = {name: evaluate_contract(runs) for name, runs in generation_runs.items()}

print(f'Measured over {len(question_contexts)} dev question(s): '
      f'{answerable_count} answerable, {unanswerable_count} unanswerable.\n')
header = ('contract', 'abstain', 'answer', 'cite', 'ground', 'workrel', 'tokens', 'p95 ms')
print('{:<12} {:>7} {:>7} {:>6} {:>6} {:>7} {:>7} {:>8}'.format(*header))
for name, m in generation_metrics.items():
    print('{:<12} {:>7.0%} {:>7.0%} {:>6.0%} {:>6.0%} {:>7.0%} {:>7.0f} {:>8.1f}'.format(
        name, m['abstention_accuracy'], m['answer_rate'], m['citation_validity'],
        m['grounding'], m['work_relevance'], m['avg_tokens'], m['p95_latency_ms']))

# Show the strict contract on a comparison answer and on an out-of-corpus refusal, visible by eye.
strict_runs = generation_runs['strict']
comparison = next((r for r in strict_runs if r['answerable'] and len(r['expected_works']) > 1), None)
if comparison is None:
    comparison = next((r for r in strict_runs if r['answerable']), None)
refusal = next((r for r in strict_runs if not r['answerable']), None)
if comparison:
    print(f'\nStrict answer - "{comparison["question"][:64]}...":')
    print('  ' + '\n  '.join(comparison['answer'].split('\n')[:8]))
if refusal:
    print(f'\nStrict on out-of-corpus - "{refusal["question"][:64]}...":')
    print(f'  {refusal["answer"][:120]}')


## 5. Justify the Selection

- **Decision rule (data-driven, deterministic):** first keep only **feasible** contracts — those with **perfect abstention accuracy** (never answer an out-of-corpus question) and **full citation validity** (every answer traces to a passage). Among the feasible, prefer **higher answer rate**, then **higher grounding**, then **higher work relevance**, then **fewer tokens**, then the **simpler** contract. The rule is applied to the measured numbers, not asserted.
- **Why abstention and citations gate everything:** a confident answer to an unanswerable question is a hallucination the pipeline exists to prevent, and an answer with no resolvable citation cannot be checked. Neither is a quality dial to trade off — they are correctness constraints, so they filter the candidates before any trade-off is weighed.
- **Why answer rate leads and simplicity breaks ties:** among contracts that are safe and traceable, the best one still answers the questions the corpus *can* support; a contract that earns perfect abstention by refusing everything is useless. When two contracts tie on answering and grounding, the simpler prompt is the honest choice.
- **Honest, data-dependent outcome (no pre-ordained winner):** closed-book cannot cite our numbered passages and does not know what is outside the corpus, so it tends to **fail both hard constraints**. Permissive grounding answers well but rarely abstains and often skips citations, so it tends to **fail abstention or citation validity**. Strict grounding is built to clear both — but only the measured numbers decide, and if strictness over-refuses answerable questions its low answer rate is recorded against it. We report whichever the numbers choose.
- **Expected result:** the selected contract with the numbers that justify it, and each rejected alternative annotated with its measured reason.


In [ ]:
SIMPLICITY_ORDER = {'closed-book': 0, 'permissive': 1, 'strict': 2}   # tie-break; lower = simpler prompt


def is_feasible(name):
    m = generation_metrics[name]
    # Hard constraints: never answer an out-of-corpus question, and cite every answer that is given.
    return m['abstention_accuracy'] >= 1.0 - 1e-9 and m['citation_validity'] >= 1.0 - 1e-9


def selection_key(name):
    m = generation_metrics[name]
    # Among feasible contracts: more answers, then more grounding, then more relevance, then fewer tokens, then simpler.
    return (-m['answer_rate'], -m['grounding'], -m['work_relevance'], m['avg_tokens'], SIMPLICITY_ORDER.get(name, 99))


feasible = [name for name in generation_metrics if is_feasible(name)]
if feasible:
    selected_contract = min(feasible, key=selection_key)
else:
    # Nothing cleared the constraints: fall back to the best abstention then citation validity, and flag it.
    selected_contract = max(generation_metrics,
                            key=lambda n: (generation_metrics[n]['abstention_accuracy'],
                                           generation_metrics[n]['citation_validity']))

selected = generation_metrics[selected_contract]
print(f'Selected contract: {selected_contract}')
print(f"  abstention / answer rate:  {selected['abstention_accuracy']:.0%} / {selected['answer_rate']:.0%}")
print(f"  citation / grounding:      {selected['citation_validity']:.0%} / {selected['grounding']:.0%}")
print(f"  work relevance / tokens:   {selected['work_relevance']:.0%} / {selected['avg_tokens']:.0f}")
print('\nAlternatives:')
for name, m in generation_metrics.items():
    if name == selected_contract:
        continue
    if m['abstention_accuracy'] < 1.0 - 1e-9:
        reason = f"answers {1 - m['abstention_accuracy']:.0%} of out-of-corpus questions - disqualified"
    elif m['citation_validity'] < 1.0 - 1e-9:
        reason = f"only {m['citation_validity']:.0%} of answers cite a resolvable passage - disqualified"
    elif m['answer_rate'] < selected['answer_rate'] - 1e-9:
        reason = f"over-refuses: answers only {m['answer_rate']:.0%} of answerable questions"
    else:
        reason = 'feasible but lower grounding / relevance or more tokens than the selection'
    print(f"  {name}: abstain={m['abstention_accuracy']:.0%} cite={m['citation_validity']:.0%} "
          f"answer={m['answer_rate']:.0%} ground={m['grounding']:.0%} - {reason}")


## 6. Export the Handoff

- **What gets exported:**
  - `.artifacts/generation-handoff.json` — the decision record stages 09 and 10 load: the selected grounding contract (`closed-book` / `permissive` / `strict`), the exact system prompt text, the citation format, the abstention sentinel and rule, the answer style (fact separated from interpretation), the completion budget, the full contract comparison (abstention accuracy, answer rate, citation validity, grounding, work relevance, tokens, latency), and a back-reference to the stage 07 context decision.
- **Why a config handoff, not an answer dump:** the pipeline reproduces the context from the upstream handoffs; generation adds only the *contract* that turns a context into a grounded, cited, abstaining answer. The handoff is that reproducible configuration plus the evidence for it. Stages 09-10 replay the pipeline and this contract to reproduce or evaluate any answer.
- **The first end-to-end baseline:** this is the first stage that produces complete answers. Stage 09 decides when and how to retrieve per query; stage 10 scores these answers systematically. Both start from this contract.
- **Hygiene:** `.artifacts/` is gitignored and notebook outputs are stripped on commit — do not commit the handoff or the generation cache.
- **Expected result:** the handoff written, with the selected contract, abstention rule, and citation format printed.


In [ ]:
generation_handoff_path = ARTIFACTS / 'generation-handoff.json'

SELECTED_SYSTEM_PROMPT = GEN_CONTRACTS[selected_contract]['system']

# generation-handoff.json: the grounding contract stages 09-10 replay - how to turn a cited context into a
# grounded, abstaining answer - plus the measured comparison that justified it. Upstream handoffs hold the rest.
generation_handoff = {
    'stage': '08-grounded-generation',
    'selected_contract': selected_contract,               # closed-book | permissive | strict
    'system_prompt': SELECTED_SYSTEM_PROMPT,
    'uses_context': GEN_CONTRACTS[selected_contract]['use_context'],
    'max_completion_tokens': MAX_COMPLETION_TOKENS,
    'citation_format': '[n] referencing the numbered context passages; markers resolve to chunk_id/work/scene',
    'abstention': {'sentinel': ABSTAIN_SENTINEL, 'rule': 'emit the sentinel when the context cannot support an answer'},
    'answer_style': 'facts from passages first, each cited; interpretation marked separately',
    'selection_rule': 'feasible (abstention accuracy == 1.0, citation validity == 1.0), then max answer rate, max grounding, max work relevance, min tokens, then simplicity',
    'context_reference': {                                 # the fixed upstream this stage answers from
        'stage': context_handoff.get('stage', '07-context-building'),
        'selected_builder': SELECTED_BUILDER,
        'token_budget': CONTEXT_TOKEN_BUDGET,
        'retrieval_method': SELECTED_METHOD,
        'reranker': SELECTED_RERANKER,
        'top_k': TOP_K,
    },
    'candidates': generation_metrics,                     # abstention, answer rate, citation, grounding, relevance, cost
    'evaluated_questions': {'answerable': answerable_count, 'unanswerable': unanswerable_count},
    'production_target': {
        'module': 'production/generation/generator.py',
        'entrypoint': 'generate_answer(query, context)',
    },
}
generation_handoff_path.write_text(json.dumps(generation_handoff, ensure_ascii=False, indent=2), encoding='utf-8')

print(f'Wrote decision record to {generation_handoff_path.relative_to(ROOT)}')
print(f'Selected: {selected_contract} | abstains via "{ABSTAIN_SENTINEL}" | cites [n] passages | budget={MAX_COMPLETION_TOKENS} tokens')
print('Reminder: .artifacts/ is gitignored and notebook outputs are stripped on commit - do not commit the handoff.')


## Observations and Open Questions

Honest notes from working through this stage — what the numbers do and do not prove, and what to revisit later.

- **Generation can only be as grounded as the context allows.** Every faithfulness number here is bounded by what context building handed over. If the evidence for a question never made it into the context, the honest behavior is abstention, not a lucky answer from memory — and that is exactly what the strict contract is built to do.
- **The closed-book control earns its place by failing loudly.** It answers out-of-corpus questions confidently and cites nothing, which is the single clearest picture of why this whole pipeline exists. Keeping it in the comparison turns "grounding matters" from a claim into a measured gap.
- **Abstention and over-refusal are two sides of one dial.** Tightening the prompt to never hallucinate can push the model to refuse answerable questions too. We gate on perfect abstention but then *reward answer rate*, so a contract cannot win by refusing everything. On a larger question set this tension is where most of the tuning will live.
- **Lexical grounding is a proxy, not a verdict.** Word overlap with the context catches blatant parametric drift but misses paraphrase and cannot tell whether a citation truly supports its claim. That judgment is deferred to stage 10, which brings an LLM judge and held-out questions.
- **Caching makes the baseline reproducible.** Answers are replayed from disk, so stage 09 and stage 10 compare against a fixed baseline rather than a moving target. Re-running after a kernel restart bills nothing.
- **How we will revisit.** The contract is a swappable policy: change the system prompt, the sentinel, or the citation format, re-export the handoff, and re-run on the same questions. Tune on the development set; check the held-out set only in stage 10 so evaluation stays honest.


## Completion Check and Next Notebook

- **Restart and run in order:** restart the kernel and run top to bottom to confirm this notebook depends only on the notebook 04-07 artifacts, not hidden state. Question embedding, any replayed reranking, and generation are all served from disk caches when warm, so a clean run bills nothing.
- **What we achieved:** three grounding contracts (closed-book, permissive, strict) run over the same stage 07 contexts and graded on abstention accuracy, answer rate, citation validity, grounding, work relevance, tokens, and latency; a worked grounded answer and a worked refusal shown by eye; a deterministic, constraint-gated selection that keeps the simplest contract which never hallucinates on out-of-corpus questions, cites every answer, and still answers the corpus-supported questions; and the exported generation handoff — the first complete end-to-end baseline.
- **Honest limits:** the labeled dev set is small and the grounding metric is a lexical proxy, so these numbers pre-screen contracts rather than certify answer quality. The systematic judge is notebook 10.
- **Privacy and hygiene:** outputs are stripped on commit by `nbstripout`; `.artifacts/` is ignored. Do not commit the handoff or the generation cache.
- **Next stage — 09 Query Handling and Adaptive RAG:** load `.artifacts/generation-handoff.json`, then decide *per query* how to handle it before generation — classify intent, rewrite or decompose multi-part questions, route trivial or out-of-corpus queries away from retrieval, and adapt how much evidence to gather — so the fixed baseline built here becomes adaptive.


## Questions, Discussions, and Further Insights

The "Observations and Open Questions" cell records caveats about *our* numbers. These questions build the transferable intuition.

**Q1. What does "grounded" actually mean, and why isn't a capable model enough?**
Grounded means the answer's source of truth is the retrieved **context**, not the model's **parametric memory**. A capable model is *more* dangerous ungrounded, because it produces fluent, confident, unverifiable text. Grounding redirects the model from "recall the answer" to "read these passages and compose an answer from them," which is the only mode in which a citation means anything.

**Q2. Why compare against a closed-book control at all?**
Because it measures the thing the whole pipeline is for. Closed-book answers from memory: it cannot cite our passages and does not know which questions fall outside the ten-play corpus, so it hallucinates on the unanswerable ones. The gap between closed-book and the grounded contracts is the concrete value of retrieval and grounding, stated in numbers rather than asserted.

**Q3. Why is abstention a hard constraint instead of a scored metric?**
Because answering an unanswerable question is not "slightly worse" — it is a hallucination, the exact failure the system exists to prevent. A confident answer to "What is the population of Verona?" is wrong no matter how well written. Correctness constraints must simply hold, so abstention accuracy filters the candidate contracts before any quality trade-off is considered.

**Q4. If the prompt forces abstention, won't the model refuse good questions too?**
It can, and that is why abstention accuracy alone is not the objective. We gate on perfect abstention but then reward **answer rate** — the share of answerable questions actually answered — so a contract that refuses everything scores a perfect abstention and a terrible answer rate, and loses. The goal is the safe contract that still answers the most corpus-supported questions.

**Q5. How do we know an answer is faithful rather than just fluent?**
Here we use two cheap, deterministic proxies: **citation validity** (every bracketed marker resolves to a passage we actually provided) and **grounding** (the answer's content words overlap the context). They catch blatant drift and missing citations. They do *not* verify that a cited passage truly supports its claim — paraphrase and subtle misattribution slip through — which is why stage 10 adds an LLM judge and held-out questions.

**Q6. Where do the citations come from, and why not let the model number its own sources?**
The numbered passages and their resolvable metadata (`work`, scene, speakers, `chunk_id`) were built in stage 07. The model only references the existing numbers; it never invents citation strings. Building provenance into the context and asking the answer to point at it is what makes "cite the passage, separate fact from interpretation" enforceable rather than hoped for.

**Q7. What did we produce, and what happens next?**
We exported `.artifacts/generation-handoff.json` — the selected grounding contract, its system prompt, the citation format, the abstention rule, and the measured comparison that justified it — plus the first complete end-to-end answers. Stage 09 (Query Handling and Adaptive RAG) decides per query how to prepare and route the question before this contract runs; stage 10 (Continuous Evaluation) judges these answers systematically on the held-out set.
